# AdalFlow: A Small End-to-End LLM Application

This notebook demonstrates AdalFlow's most useful ideas in one compact project:

1. A model-agnostic `Generator`
2. Transparent Jinja-style prompts
3. Structured outputs with `DataClass` and `JsonOutputParser`
4. A lightweight Retrieval-Augmented Generation workflow
5. Evaluation over a tiny labeled dataset
6. A simple data-driven prompt-selection loop

The example builds an internal **Data Science Policy Assistant**. It retrieves relevant policy text, answers a question, returns citations, and measures answer quality.

> **Cost note:** Live cells call an external model API and may incur charges. Start with a low-cost model and a small evaluation set.

## 1. Install dependencies

Run this cell once. Restart the kernel if your environment asks you to do so.

In [1]:
%pip install -q -U adalflow openai python-dotenv scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2. Configure the model

Set `OPENAI_API_KEY` securely. The notebook asks for it without echoing it. You can replace `OpenAIClient` with another AdalFlow model client while preserving the rest of the pipeline.

In [2]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ API key: ")

MODEL_NAME = os.getenv("ADALFLOW_MODEL", "openai/gpt-oss-20b")
print("Model:", MODEL_NAME)

Model: openai/gpt-oss-20b


## 3. Imports and a basic Generator

`Generator` coordinates prompt formatting, the model call, and output processing. Its result contains fields such as `data`, `raw_response`, and `error`, which are useful for application logging.

In [3]:
import sys

print(sys.executable)
print(sys.version)

%pip uninstall -y adalflow
%pip install --no-cache-dir --force-reinstall "adalflow==1.1.3"
%pip install --no-cache-dir --upgrade groq python-dotenv

e:\Auto-Analyst-Autonomous-Assistant\venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Found existing installation: adalflow 1.1.3
Uninstalling adalflow-1.1.3:
  Successfully uninstalled adalflow-1.1.3
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 5.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   --------------------------------- ------ 1.0/1.3 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 6.3 MB/s  0:00:00
   ---------------------------------------- 0.0/944.4 kB ? eta -:--:--
   ---------------------------------------- 944.4/944.4 kB 5.5 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.3 requires groq<1.0.0,>=0.30.0, but you have groq 1.7.0 which is incompatible.


In [5]:
import os
from getpass import getpass
import adalflow as adal
from adalflow.utils import setup_env

# Enter the key securely if it is not already configured
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

MODEL_NAME = "openai/gpt-oss-20b"

basic_llm = adal.Generator(
    model_client=adal.GroqAPIClient(),
    model_kwargs={
        "model": MODEL_NAME,
        "temperature": 0,
    },
)

result = basic_llm(
    prompt_kwargs={
        "input_str": "Explain RAG in one sentence."
    }
)

print("Data:", result.data)
print("Error:", result.error)

Data: RAG (Retrieval‑Augmented Generation) is a technique that first retrieves relevant documents from a knowledge base and then feeds them into a language model to generate more accurate, context‑aware responses.
Error: None


## 4. Create a tiny knowledge base

In production, these passages might come from SharePoint, a vector database, PDFs, or a data catalog. Here we keep everything local so the retrieval logic is visible.

In [6]:
DOCUMENTS = [
    {
        "id": "POL-001",
        "title": "Model Review",
        "text": "Any model used for a customer-facing decision must complete fairness, privacy, security, and performance reviews before production deployment."
    },
    {
        "id": "POL-002",
        "title": "PII Handling",
        "text": "Personally identifiable information must not be copied into public AI services. Approved private endpoints and documented access controls are required."
    },
    {
        "id": "POL-003",
        "title": "Monitoring",
        "text": "Production models must be monitored for data drift, prediction drift, latency, failure rate, and business impact. Owners review alerts at least weekly."
    },
    {
        "id": "POL-004",
        "title": "Experiment Reproducibility",
        "text": "Experiments must record the dataset version, code commit, model configuration, random seed, evaluation metrics, and responsible owner."
    },
    {
        "id": "POL-005",
        "title": "Human Oversight",
        "text": "High-impact automated recommendations require a documented human review path and an escalation process for disputed outcomes."
    },
]

len(DOCUMENTS)

5

## 5. Add a transparent local retriever

This retriever uses TF-IDF and cosine similarity. It is intentionally simple. AdalFlow can also be connected to dedicated retrievers and vector stores.

In [7]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class LocalRetriever:
    def __init__(self, documents):
        self.documents = documents
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        corpus = [f"{d['title']} {d['text']}" for d in documents]
        self.matrix = self.vectorizer.fit_transform(corpus)

    def retrieve(self, query, top_k=3):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix)[0]
        ranked = scores.argsort()[::-1][:top_k]
        return [
            {**self.documents[i], "score": float(scores[i])}
            for i in ranked
        ]

retriever = LocalRetriever(DOCUMENTS)
retriever.retrieve("How should we monitor a deployed model?", top_k=2)

[{'id': 'POL-001',
  'title': 'Model Review',
  'text': 'Any model used for a customer-facing decision must complete fairness, privacy, security, and performance reviews before production deployment.',
  'score': 0.30118139355878737},
 {'id': 'POL-004',
  'title': 'Experiment Reproducibility',
  'text': 'Experiments must record the dataset version, code commit, model configuration, random seed, evaluation metrics, and responsible owner.',
  'score': 0.14572725039582135}]

## 6. Define a structured response

The model must return an answer, cited document IDs, confidence, and a follow-up action. AdalFlow's parser turns valid JSON into the `PolicyAnswer` data class.

In [9]:
from dataclasses import dataclass, field
from adalflow.core import DataClass
from adalflow.components.output_parsers import JsonOutputParser

@dataclass
class PolicyAnswer(DataClass):
    answer: str = field(metadata={"desc": "A concise answer grounded only in the supplied context."})
    citations: list[str] = field(metadata={"desc": "Document IDs supporting the answer, for example POL-003."})
    confidence: float = field(metadata={"desc": "A number from 0.0 to 1.0."})
    next_action: str = field(metadata={"desc": "One practical action for the user."})

parser = JsonOutputParser(data_class=PolicyAnswer, return_data_class=True)
print(parser.format_instructions())

Your output should be formatted as a standard JSON instance with the following schema:
```
{
    "type": "PolicyAnswer",
    "properties": {
        "answer": {
            "type": "str",
            "desc": "A concise answer grounded only in the supplied context."
        },
        "citations": {
            "type": "List[str]",
            "desc": "Document IDs supporting the answer, for example POL-003."
        },
        "confidence": {
            "type": "float",
            "desc": "A number from 0.0 to 1.0."
        },
        "next_action": {
            "type": "str",
            "desc": "One practical action for the user."
        }
    },
    "required": [
        "answer",
        "citations",
        "confidence",
        "next_action"
    ]
}
```
**Schema Interpretation:**
   - The "properties" and "type" fields in the schema are NOT the actual JSON keys
   - Generate the correct nested JSON structure using the actual field names shown
   - Follow the exact field names

## 7. Build the RAG component

The prompt is fully visible and editable. The model is explicitly told to use only retrieved context and to admit when evidence is insufficient.

In [11]:
RAG_TEMPLATE = r"""
<SYSTEM>
You are a careful internal policy assistant.

Rules:
1. Answer using only CONTEXT.
2. Do not invent policies or citations.
3. If context is insufficient, say so and lower confidence.
4. Keep the answer concise and operational.
5. Return exactly the required structured format.

<OUTPUT_FORMAT>
{{ output_format_str }}
</OUTPUT_FORMAT>
</SYSTEM>

<CONTEXT>
{{ context }}
</CONTEXT>

<QUESTION>
{{ question }}
</QUESTION>
"""

class PolicyRAG(adal.Component):
    def __init__(self, retriever, template=RAG_TEMPLATE):
        super().__init__()
        self.retriever = retriever
        self.generator = adal.Generator(
            model_client=adal.GroqAPIClient(),
            model_kwargs={"model": MODEL_NAME, "temperature": 0},
            template=template,
            prompt_kwargs={"output_format_str": parser.format_instructions()},
            output_processors=parser,
        )

    def call(self, question: str, top_k: int = 3):
        docs = self.retriever.retrieve(question, top_k=top_k)
        context = "\n\n".join(
            f"[{d['id']}] {d['title']}: {d['text']}"
            for d in docs
        )
        response = self.generator(
            prompt_kwargs={"question": question, "context": context}
        )
        return response, docs

assistant = PolicyRAG(retriever)

## 8. Ask a grounded question

In [12]:
question = "What must I record so another data scientist can reproduce my experiment?"
response, retrieved_docs = assistant(question)

print("Retrieved:")
for doc in retrieved_docs:
    print(f"  {doc['id']} | score={doc['score']:.3f} | {doc['title']}")

print("\nStructured answer:")
print(response.data)
print("\nParser/model error:", response.error)

Retrieved:
  POL-004 | score=0.209 | Experiment Reproducibility
  POL-003 | score=0.099 | Monitoring
  POL-005 | score=0.000 | Human Oversight

Structured answer:
PolicyAnswer(answer='Record the dataset version, code commit, model configuration, random seed, evaluation metrics, and the responsible owner.', citations=['POL-004'], confidence=1.0, next_action='Create a reproducibility log template that includes fields for dataset version, code commit, model configuration, random seed, evaluation metrics, and owner.')

Parser/model error: None


## 9. Add safety-oriented application checks

Structured generation is useful, but production code should still validate the result. This function verifies that citations really came from retrieved documents and that confidence is in range.

In [13]:
def validate_answer(response, retrieved_docs):
    if response.error:
        return False, [f"Generator error: {response.error}"]
    if response.data is None:
        return False, ["No parsed data returned"]

    issues = []
    allowed_ids = {d["id"] for d in retrieved_docs}
    cited_ids = set(response.data.citations)

    if not cited_ids.issubset(allowed_ids):
        issues.append(f"Unsupported citations: {sorted(cited_ids - allowed_ids)}")
    if not 0.0 <= response.data.confidence <= 1.0:
        issues.append("Confidence is outside [0, 1]")
    if not response.data.answer.strip():
        issues.append("Answer is empty")

    return len(issues) == 0, issues

is_valid, issues = validate_answer(response, retrieved_docs)
print("Valid:", is_valid)
print("Issues:", issues)

Valid: True
Issues: []


## 10. Evaluate the pipeline

This small labeled set checks two things:

- **Retrieval recall:** Was the expected source retrieved?
- **Citation accuracy:** Did the structured answer cite the expected source?

A real project should add semantic answer quality, groundedness, latency, cost, and human review metrics.

In [14]:
EVAL_SET = [
    ("What reviews are needed before a customer-facing model goes live?", "POL-001"),
    ("Can I paste customer PII into a public AI website?", "POL-002"),
    ("Which signals must be monitored after deployment?", "POL-003"),
    ("How do I make an experiment reproducible?", "POL-004"),
    ("What oversight is required for high-impact recommendations?", "POL-005"),
]

def evaluate(app, eval_set=EVAL_SET, top_k=3):
    rows = []
    for question, expected_id in eval_set:
        response, docs = app(question, top_k=top_k)
        retrieved_ids = [d["id"] for d in docs]
        cited_ids = response.data.citations if response.data else []
        valid, issues = validate_answer(response, docs)
        rows.append({
            "question": question,
            "expected_id": expected_id,
            "retrieved_ids": retrieved_ids,
            "cited_ids": cited_ids,
            "retrieval_hit": expected_id in retrieved_ids,
            "citation_hit": expected_id in cited_ids,
            "valid": valid,
            "issues": issues,
        })
    return rows

rows = evaluate(assistant)

for row in rows:
    print({k: row[k] for k in ["expected_id", "retrieval_hit", "citation_hit", "valid"]})

retrieval_recall = sum(r["retrieval_hit"] for r in rows) / len(rows)
citation_accuracy = sum(r["citation_hit"] for r in rows) / len(rows)
valid_rate = sum(r["valid"] for r in rows) / len(rows)

print(f"Retrieval recall: {retrieval_recall:.0%}")
print(f"Citation accuracy: {citation_accuracy:.0%}")
print(f"Valid-output rate: {valid_rate:.0%}")

{'expected_id': 'POL-001', 'retrieval_hit': True, 'citation_hit': True, 'valid': True}
{'expected_id': 'POL-002', 'retrieval_hit': True, 'citation_hit': True, 'valid': True}
{'expected_id': 'POL-003', 'retrieval_hit': True, 'citation_hit': True, 'valid': True}
{'expected_id': 'POL-004', 'retrieval_hit': True, 'citation_hit': True, 'valid': True}
{'expected_id': 'POL-005', 'retrieval_hit': True, 'citation_hit': True, 'valid': True}
Retrieval recall: 100%
Citation accuracy: 100%
Valid-output rate: 100%


## 11. Data-driven prompt optimization, kept simple

AdalFlow supports trainable prompt parameters and optimizer-driven workflows. To keep this notebook short and robust, the following cell demonstrates the core optimization idea with prompt candidates: evaluate each prompt on labeled examples, then retain the best-performing version.

This is deliberately a tiny search loop, not a substitute for AdalFlow's full trainer and textual-gradient optimization APIs.

In [15]:
STRICT_TEMPLATE = RAG_TEMPLATE.replace(
    "5. Return exactly the required structured format.",
    "5. Return exactly the required structured format. Every factual sentence must be supported by at least one retrieved document ID."
)

CONCISE_TEMPLATE = RAG_TEMPLATE.replace(
    "Keep the answer concise and operational.",
    "Use no more than three sentences and make the first sentence the direct answer."
)

candidates = {
    "baseline": RAG_TEMPLATE,
    "strict_citations": STRICT_TEMPLATE,
    "concise": CONCISE_TEMPLATE,
}

scores = {}
for name, template in candidates.items():
    candidate_app = PolicyRAG(retriever, template=template)
    candidate_rows = evaluate(candidate_app)
    citation_score = sum(r["citation_hit"] for r in candidate_rows) / len(candidate_rows)
    valid_score = sum(r["valid"] for r in candidate_rows) / len(candidate_rows)
    scores[name] = 0.8 * citation_score + 0.2 * valid_score
    print(name, round(scores[name], 3))

best_name = max(scores, key=scores.get)
print("Best prompt:", best_name)
optimized_assistant = PolicyRAG(retriever, template=candidates[best_name])

baseline 1.0


Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'


strict_citations 0.0


Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'
Error in converting dict to data class: 'type'
Error processing the output processors: 'type'


concise 0.0
Best prompt: baseline


In [16]:
assistant = PolicyRAG(retriever)

rows = evaluate(assistant)

retrieval_recall = sum(r["retrieval_hit"] for r in rows) / len(rows)
citation_accuracy = sum(r["citation_hit"] for r in rows) / len(rows)
valid_rate = sum(r["valid"] for r in rows) / len(rows)

print(f"Retrieval recall: {retrieval_recall:.0%}")
print(f"Citation accuracy: {citation_accuracy:.0%}")
print(f"Valid-output rate: {valid_rate:.0%}")

Retrieval recall: 100%
Citation accuracy: 100%
Valid-output rate: 100%


## 12. Try an out-of-scope question

A grounded application should refuse to manufacture an answer when the knowledge base does not contain the required information.

In [17]:
response, docs = optimized_assistant("What is the company's annual leave allowance?")
print(response.data)
print("Retrieved IDs:", [d["id"] for d in docs])

Error in converting dict to data class: 'type'
Error processing the output processors: 'type'


None
Retrieved IDs: ['POL-005', 'POL-004', 'POL-003']


## What this notebook demonstrates

- **Composition:** Retrieval and generation are separate, replaceable components.
- **Model portability:** Change the model client and model configuration without rewriting application logic.
- **Prompt transparency:** The full prompt is explicit and inspectable.
- **Structured outputs:** Responses become typed Python objects rather than unvalidated strings.
- **Evaluation:** The application is tested against labeled examples.
- **Optimization mindset:** Prompts are selected using measured performance, not intuition alone.
- **Production checks:** Citations and confidence are validated outside the model.

## Next steps

1. Replace TF-IDF with embeddings and a vector store.
2. Load real documents from an approved enterprise source.
3. Add tracing, latency, token usage, and cost measurements.
4. Expand the evaluation set and add human-reviewed groundedness scores.
5. Explore AdalFlow's `Parameter`, `Trainer`, and optimizer tutorials for automatic prompt and few-shot optimization.

## References

- AdalFlow documentation: https://adalflow.sylph.ai/
- Generator tutorial: https://adalflow.sylph.ai/new_tutorials/generator.html
- GitHub repository: https://github.com/SylphAI-Inc/AdalFlow